# RAG-Powered FAQ Agent with Custom Knowledge

Answers customer questions from a custom FAQ document using retrieval-augmented generation (RAG).


| Stage | Resource |
|---|---|
| Embeddings | **Local** `all-MiniLM-L6-v2` (free, no API key) |
| LLM | Standard **OpenAI** or **Anthropic** API key |
| Vector store | FAISS |

**Setup:**
1. Add `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` to Colab **Secrets** (🔑) with notebook access on.
2. Upload `faq.txt` to the Colab file browser.
3. Set `PROVIDER` in Step 5.

## Step 1: Install packages

In [1]:
!pip install -q -U langchain langchain-classic langchain-community langchain-text-splitters langchain-huggingface sentence-transformers faiss-cpu langchain-openai langchain-anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.4/170.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.4/146.4 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.8/573.8 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.5/862.5 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

## Step 2: Import dependencies

In [2]:
import os

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

/tmp/ipykernel_2080/430979806.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


## Step 3: Load and chunk the FAQ document

In [3]:
loader = TextLoader("faq.txt")   # upload faq.txt to the Colab file browser first
documents = loader.load()

text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)
print(f"Loaded {len(documents)} document(s) and split into {len(docs)} chunks.")

Loaded 1 document(s) and split into 4 chunks.


## Step 4: Embed the chunks locally and build the FAISS index

The embedding model runs on the Colab machine: no API key and no cost. It downloads once (about 90 MB).

In [4]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)
print(f"Indexed {len(docs)} chunks in FAISS.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexed 4 chunks in FAISS.


## Step 5: Connect to the LLM
Set `PROVIDER` to `"openai"` or `"anthropic"`, matching the key you added to Secrets.

In [5]:
def _load(key):
  """Colab Secrets first, then an environment variable."""
  try:
    from google.colab import userdata
    value = userdata.get(key)
    if value:
      return value
  except Exception:
    pass
  return os.environ.get(key)

PROVIDER = "openai"   # or "anthropic"

if PROVIDER == "openai":
  from langchain_openai import ChatOpenAI
  llm = ChatOpenAI(model="gpt-5-mini", api_key=_load("OPENAI_API_KEY"))
elif PROVIDER == "anthropic":
  from langchain_anthropic import ChatAnthropic
  llm = ChatAnthropic(model="claude-haiku-5-5", api_key=_load("ANTHROPIC_API_KEY"), max_tokens=1024)
else:
  raise ValueError('PROVIDER must be "openai" or "anthropic"')

print(f"Using {PROVIDER}")

Using openai


## Step 6: Create the RAG chain

The prompt tells the model to answer **only** from the retrieved FAQ text and to say so when the answer isn't there, rather than guessing.

In [6]:
PROMPT = PromptTemplate.from_template(
"""You are a customer support assistant. Answer the question using ONLY the FAQ context below.
  If the answer is not in the context, say "I couldn't find that in our FAQ" and suggest contacting support.

  FAQ context:
  {context}

  Question: {question}
  Answer:
"""
)

qa_chain = RetrievalQA.from_chain_type(
  llm=llm,
  retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
  return_source_documents=True,
  chain_type_kwargs={"prompt": PROMPT},
)

## Step 7: Ask questions

The last question is deliberately outside the FAQ, to check that the agent declines instead of making something up.

In [7]:
questions = [
  "What is our return policy?", # Answered Directly in Source
  "Can I cancel my order after placing it?", # Answered indrectly in Source
  "Do you offer a student discount on enterprise plans?", # Not addressed in source
]

for query in questions:
  result = qa_chain.invoke({"query": query})
  print("=" * 70)
  print("Q:", query)
  print("A:", result["result"])
  print("\nSources:")
  for i, doc in enumerate(result["source_documents"], 1):
    print(f"  [{i}] {doc.page_content[:150].strip()}...")

Q: What is our return policy?
A: We offer a 30-day return policy for all unopened items. Products must be returned in their original packaging with proof of purchase.

Sources:
  [1] Q: What is your return policy?
A: We offer a 30-day return policy for all unopened items. Products must be returned in original packaging with proof o...
  [2] Q: Can I track my order?
A: Yes, once your order is shipped, you will receive a tracking link via email and SMS.

Q: What should I do if I receive a d...
  [3] Q: What payment methods are accepted?
A: We accept Visa, MasterCard, American Express, PayPal, and UPI payments.

Q: How can I cancel my order?
A: Ord...
Q: Can I cancel my order after placing it?
A: Yes. Orders can be cancelled within 2 hours of placement. To cancel, log in to your account, go to "My Orders", and click "Cancel Order".

Sources:
  [1] Q: What payment methods are accepted?
A: We accept Visa, MasterCard, American Express, PayPal, and UPI payments.

Q: How can I cancel my order?